# PyCytoMiner workflow
As described in https://www.ncbi.nlm.nih.gov/pmc/articles/PMC10516049/

In [ ]:
import os
import pandas as pd
import numpy as np
import scanpy as sc
import anndata
from tqdm.auto import tqdm
from concurrent.futures import ProcessPoolExecutor, as_completed
import pycytominer

DMSO_INCHIKEY="IAZDPXIOMUYVGZ-UHFFFAOYSA-N"

In [ ]:
def summarize_missing_values(df: pd.DataFrame) -> pd.DataFrame:
    """
    Summarize NaN, positive infinity, and negative infinity occurrences in a DataFrame.

    :param df: pandas DataFrame to summarize.
    :return: pandas DataFrame summarizing NaN, positive infinity, and negative infinity counts.
    """
    nan_counts = df.isna().sum().sum()
    pos_inf_counts = (df == np.inf).sum().sum()
    neg_inf_counts = (df == -np.inf).sum().sum()

    summary_df = pd.DataFrame({
        "NaN_count": [nan_counts],
        "Pos_inf_count": [pos_inf_counts],
        "Neg_inf_count": [neg_inf_counts]
    })

    return summary_df


In [ ]:
target2_complete = pd.read_parquet("../data/target2_wellres_featuresimputed_druginfoadded.parquet")

summarize_missing_values(target2_complete)

## Drop per-image (`Image_*`) features

pycytominer >= 1.5.0 passes `Image_*` columns through `normalize()` un-normalized
(the OME-Arrow payload passthrough added in PR #643). CellProfiler uses `Image_`
for per-image *measurements*, so 1089 raw features would otherwise survive every
`feature_select` filter. Dropping them reproduces the original 591 features.

In [ ]:
target2_complete = target2_complete.drop(
    columns=[c for c in target2_complete.columns if c.startswith("Image_")]
)

metadata_cols = [col for col in target2_complete.columns if "Metadata" in col]

target2_complete.sort_values(["Metadata_Source", "Metadata_Plate", "Metadata_Well"], inplace=True)
target2_complete.index = [f"{row['Metadata_Source']}__{row['Metadata_Plate']}__{row['Metadata_Well']}" for _, row in target2_complete.iterrows()]

target2_complete

## MAD robustize per plate, based on DMSO

In [ ]:
%%time

def transform_group(group_details):
    group, metadata_cols = group_details
    group_transformed = pycytominer.normalize(
        profiles=group,
        features="infer",
        meta_features=metadata_cols,
        samples="Metadata_InChIKey_standardized == 'IAZDPXIOMUYVGZ-UHFFFAOYSA-N'",  # DMSO
        method="mad_robustize",
    )
    return group_transformed

target2_groups = target2_complete.groupby(["Metadata_Source", "Metadata_Plate"])
group_details = [(group, metadata_cols) for name, group in target2_groups]

# Use ProcessPoolExecutor to parallelize
results = []
with ProcessPoolExecutor() as executor:
    futures = [executor.submit(transform_group, detail) for detail in group_details]
    
    for future in tqdm(as_completed(futures), total=len(futures)):
        result = future.result()
        results.append(result)

target2_norm = pd.concat(results).drop_duplicates()

## Feature select

In [ ]:
%%time

feature_select_opts = [
    "variance_threshold",
    "drop_na_columns",
    "correlation_threshold",
    "blocklist",
    "drop_outliers"
]

target2_norm_featureselected = pycytominer.feature_select(
    profiles=target2_norm,
    features="infer",
    samples="all",
    operation=feature_select_opts
)
target2_norm_featureselected

## Spherize all samples together, based on DMSO

In [ ]:
%%time

target2_norm_featureselected_spherized = pycytominer.normalize(
    profiles=target2_norm_featureselected,
    features="infer",
    meta_features=metadata_cols,
    samples=f"Metadata_InChIKey_standardized == '{DMSO_INCHIKEY}'",
    method="spherize",
).drop_duplicates()
target2_norm_featureselected_spherized

In [ ]:
adata = anndata.AnnData(X=target2_norm_featureselected_spherized.drop(metadata_cols, axis=1))

adata

## Add metadata

In [ ]:
metadata = target2_norm_featureselected_spherized[metadata_cols]
adata.obs = adata.obs.merge(metadata, left_index=True, right_index=True)
adata

## Preview features

In [ ]:
sc.pp.neighbors(adata)
sc.tl.umap(adata)

In [ ]:
sc.pl.embedding(
    adata,
    "X_umap",
    color="Metadata_Source"
)

In [ ]:
adata.write("../data/target2_wellres_featuresimputed_druginfoadded_pycytominer.h5ad")

In [ ]:
adata.obs["Metadata_compound"] = "Other"
adata.obs.loc[adata.obs["Metadata_InChIKey_standardized"] == "IAZDPXIOMUYVGZ-UHFFFAOYSA-N", "Metadata_compound"] = "DMSO"
adata.obs.loc[adata.obs["Metadata_InChIKey"] == "IVUGFMLRJOCGAS-UHFFFAOYSA-N", "Metadata_compound"] = "AMG900"
adata.obs.loc[adata.obs["Metadata_InChIKey"] == "OINGHOPGNMYCAB-INIZCTEOSA-N", "Metadata_compound"] = "NVS-PAK1-1"
adata.obs.loc[adata.obs["Metadata_InChIKey"] == "UREBDLICKHMUKA-CXSFZGCWSA-N", "Metadata_compound"] = "dexamethasone"
adata.obs.loc[adata.obs["Metadata_InChIKey"] == "IHLVSLOZUHKNMQ-UHFFFAOYSA-N", "Metadata_compound"] = "LY2109761"
adata.obs.loc[adata.obs["Metadata_InChIKey"] == "KPBNHDGDUADAGP-VAWYXSNFSA-N", "Metadata_compound"] = "FK-866"
adata.obs.loc[adata.obs["Metadata_InChIKey"] == "LOUPRKONTZGTKE-LHHVKLHASA-N", "Metadata_compound"] = "quinidine"
adata.obs.loc[adata.obs["Metadata_InChIKey"] == "CQKBSRPVZZLCJE-UHFFFAOYSA-N", "Metadata_compound"] = "TC-S-7004"
adata.obs.loc[adata.obs["Metadata_InChIKey"] == "SRVFFFJZQVENJC-IHRRRGAJSA-N", "Metadata_compound"] = "aloxistatin"

adata.obs["Metadata_compound"] = adata.obs["Metadata_compound"].astype("category")

sc.pl.embedding(
    adata,
    "X_umap",
    color="Metadata_compound",
    groups=["DMSO", "AMG900", "NVS-PAK1-1", "dexamethasone", "LY2109761", "FK-866", "quinidine", "TC-S-7004", "aloxistatin"],
    s=10,
    show=False,
    sort_order=True,
    frameon=False,
)

In [ ]:
adata.obs["Metadata_moa"] = adata.obs["Metadata_moa"].astype("category")
examples = ["CDC inhibitor", "caspase inhibitor", "integrin antagonist", "microtubule inhibitor", "protease inhibitor"]

# filter to moas to be shown so scanpy doesn't default to grey due to too many categories
adata.obs["tmp"] = [moa if moa in examples else "Other" for moa in adata.obs["Metadata_moa"].values]

sc.pl.embedding(
    adata,
    "X_umap",
    color="tmp",
    groups=examples,
    cmap="tab10",
    s=20,
    frameon=False,
)

In [ ]:
adata.obs

In [ ]:
adata.obs['Metadata_Plate'].unique()

In [ ]:
x = pd.crosstab(adata.obs.Metadata_Plate, adata.obs.Metadata_Well)